In [0]:
from pyspark.sql.functions import current_timestamp

catalog = "workspace"
schema = "lakehouse_f1"
volume_origem = "f1_raw_data"

path_origem = f"/Volumes/{catalog}/{schema}/{volume_origem}/"


In [0]:
tabelas = ["races", "lap_times", "pit_stops", "drivers", "circuits", "results", "driver_standings", "status"]

for tabela in tabelas:
    print(f"Processando arquivo: {tabela}.csv")
    
    df = spark.read.csv(f"{path_origem}{tabela}.csv", header=True, inferSchema=True)
    
    # 2. Adição de metadado (rastreabilidade de quando o dado foi carregado)
    df_bronze = df.withColumn("ingestion_date", current_timestamp())
    
    # 3. Escrita na Camada Bronze como uma Tabela Delta gerenciada
    nome_tabela_destino = f"{catalog}.{schema}.bronze_{tabela}"
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_destino)
    
print("Carga da Camada Bronze concluída com sucesso! 🏁")